# Mango Maturity Detector - Improved Training
Perbaikan dari versi sebelumnya:
1. Tambah kelas `non_mango` agar model tahu benda bukan mangga
2. Fine-tune top layers MobileNetV2 (bukan hanya head)
3. Dropout, EarlyStopping, ReduceLROnPlateau
4. Augmentasi data lebih agresif
5. Two-phase training untuk hasil optimal

In [ ]:
!pip install roboflow tensorflow-datasets -q

## 1. Download Dataset Mangga dari Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="ARGmchr8KD8vEnsId2Ks")
project = rf.workspace("ml-z0o30").project("mango-sjvso")
version = project.version(3)
dataset = version.download("folder")

print(f"Dataset tersimpan di: {dataset.location}")

## 2. Tambah Kelas Non-Mango

Ini adalah kunci perbaikan utama. Tanpa kelas ini, model tidak bisa membedakan mangga vs bukan mangga.
Kita akan download beberapa gambar dari TFDS (CIFAR-10) sebagai contoh benda bukan mangga.

In [ ]:
import os
import numpy as np
from PIL import Image
import tensorflow_datasets as tfds

# Buat folder non_mango di train dan valid
train_non_mango_dir = os.path.join(dataset.location, 'train', 'non_mango')
valid_non_mango_dir = os.path.join(dataset.location, 'valid', 'non_mango')
os.makedirs(train_non_mango_dir, exist_ok=True)
os.makedirs(valid_non_mango_dir, exist_ok=True)

# Load CIFAR-10 — ambil gambar yang jelas bukan buah (airplane, car, ship, truck, horse, deer)
# Kita filter kelas yang paling berbeda dari mangga
non_fruit_classes = [0, 1, 8, 9, 7, 5]  # airplane, automobile, ship, truck, horse, deer

ds_train, ds_valid = tfds.load('cifar10', split=['train', 'test'], as_supervised=True)

def save_non_mango_images(dataset, target_dir, max_per_class=40):
    count_per_class = {c: 0 for c in non_fruit_classes}
    total_saved = 0
    for img_tensor, label in dataset:
        label_int = int(label.numpy())
        if label_int in non_fruit_classes:
            if count_per_class[label_int] < max_per_class:
                img_array = img_tensor.numpy()
                img_pil = Image.fromarray(img_array.astype(np.uint8))
                img_pil = img_pil.resize((224, 224), Image.LANCZOS)
                save_path = os.path.join(target_dir, f"nonmango_c{label_int}_{count_per_class[label_int]}.jpg")
                img_pil.save(save_path)
                count_per_class[label_int] += 1
                total_saved += 1
    return total_saved

print("Menyimpan gambar non-mango untuk training...")
n_train = save_non_mango_images(ds_train, train_non_mango_dir, max_per_class=50)
print(f"  Tersimpan {n_train} gambar ke train/non_mango/")

print("Menyimpan gambar non-mango untuk validasi...")
n_valid = save_non_mango_images(ds_valid, valid_non_mango_dir, max_per_class=20)
print(f"  Tersimpan {n_valid} gambar ke valid/non_mango/")

print("\nSelesai! Distribusi kelas sekarang:")
train_dir = os.path.join(dataset.location, 'train')
for cls in sorted(os.listdir(train_dir)):
    n = len(os.listdir(os.path.join(train_dir, cls)))
    print(f"  {cls}: {n} gambar")

## 3. Setup Data Generator dengan Augmentasi Kuat

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

# Augmentasi agresif untuk training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='reflect'
)

# Validasi hanya rescale
val_datagen = ImageDataGenerator(rescale=1./255)

train_dir = os.path.join(dataset.location, 'train')
valid_dir = os.path.join(dataset.location, 'valid')

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    valid_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes
print(f"\nJumlah kelas: {NUM_CLASSES}")
print(f"Kelas: {train_gen.class_indices}")

## 4. Hitung Class Weights (untuk dataset tidak seimbang)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_gen.classes
)
class_weights = dict(enumerate(class_weights_array))

print("Class weights:")
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
for idx, weight in class_weights.items():
    print(f"  {idx_to_class[idx]}: {weight:.3f}")

## 5. Bangun Model

**Phase 1**: Latih hanya head (base frozen)  
**Phase 2**: Fine-tune top 40 layer MobileNetV2 dengan learning rate kecil

In [ ]:
from tensorflow.keras import layers, models, regularizers

def build_model(num_classes, fine_tune_at=None):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    if fine_tune_at is None:
        base_model.trainable = False
    else:
        base_model.trainable = True
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

model, base_model = build_model(NUM_CLASSES)
model.summary()

## 6. Phase 1 — Latih Head (Base Frozen)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks_phase1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model_phase1.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("=== Phase 1: Training Head ===")
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_phase1
)

print(f"\nBest val_accuracy Phase 1: {max(history1.history['val_accuracy']):.4f}")

## 7. Phase 2 — Fine-tune Top 40 Layer MobileNetV2

In [ ]:
# Unfreeze top 40 layer dari base model
FINE_TUNE_AT = len(base_model.layers) - 40
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Layer yang di-fine-tune: {trainable_count} dari {len(base_model.layers)}")

callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model_phase2.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Learning rate jauh lebih kecil untuk fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("=== Phase 2: Fine-tuning ===")
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks_phase2
)

print(f"\nBest val_accuracy Phase 2: {max(history2.history['val_accuracy']):.4f}")

## 8. Evaluasi pada Test Set

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Test set evaluation
test_dir = os.path.join(dataset.location, 'test')
test_datagen = ImageDataGenerator(rescale=1./255)

if os.path.exists(test_dir) and len(os.listdir(test_dir)) > 0:
    test_gen = test_datagen.flow_from_directory(
        test_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    
    # Prediksi
    y_pred_prob = model.predict(test_gen)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())
    
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()
else:
    print("Test set tidak ditemukan, evaluasi pada val set.")
    val_gen.reset()
    y_pred_prob = model.predict(val_gen)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = val_gen.classes
    class_names = list(val_gen.class_indices.keys())
    print(classification_report(y_true, y_pred, target_names=class_names))

## 9. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gabungkan history
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
phase1_end = len(history1.history['accuracy'])

axes[0].plot(acc, label='Train Accuracy')
axes[0].plot(val_acc, label='Val Accuracy')
axes[0].axvline(x=phase1_end, color='gray', linestyle='--', label='Fine-tune starts')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].set_xlabel('Epoch')

axes[1].plot(loss, label='Train Loss')
axes[1].plot(val_loss, label='Val Loss')
axes[1].axvline(x=phase1_end, color='gray', linestyle='--', label='Fine-tune starts')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 10. Export ke TFLite + Simpan Labels

In [ ]:
# Konversi ke TFLite dengan optimasi ukuran
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimasi untuk ukuran lebih kecil (kompatibel dengan tflite_flutter)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open('model_mangga.tflite', 'wb') as f:
    f.write(tflite_model)

# Simpan labels — urutan sesuai class_indices
# PENTING: urutan harus sama persis dengan yang dipakai saat training
idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
labels_ordered = [idx_to_class[i] for i in range(NUM_CLASSES)]

with open('labels.txt', 'w') as f:
    f.write('\n'.join(labels_ordered))

print(f"Model disimpan: model_mangga.tflite ({len(tflite_model) / 1024:.1f} KB)")
print(f"\nLabel kelas (urutan index):")
for i, label in enumerate(labels_ordered):
    print(f"  {i}: {label}")

## 11. Verifikasi Model TFLite

Test apakah model TFLite bisa dijalankan dan outputnya masuk akal.

In [ ]:
import numpy as np

# Load dan test model TFLite
interpreter = tf.lite.Interpreter(model_path='model_mangga.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]['shape'])    # harus [1, 224, 224, 3]
print("Input dtype:", input_details[0]['dtype'])    # harus float32
print("Output shape:", output_details[0]['shape'])  # harus [1, NUM_CLASSES]

# Test dengan input acak
dummy_input = np.random.rand(1, 224, 224, 3).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], dummy_input)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
print("\nOutput probabilities (dummy input):", output[0])
print("Sum:", output[0].sum())  # harus ~1.0
print("\nModel TFLite siap digunakan!")
print("Download: model_mangga.tflite dan labels.txt")